In [40]:
# Install openrouteservice for first time users
# !pip install openrouteservice

In [41]:
import openrouteservice
import pandas as pd
import numpy as np
from tqdm import tqdm

In [42]:
# Load in rental data
rental = pd.read_csv("rea_data/domain/Data/vic_rentals_all_cleaned.csv")
rental_coords = rental[["listing_id", "lon","lat"]]
rental_coords.head()

,listing_id,lon,lat
0,16782629,144.87091,-37.830982
1,17471867,144.86755,-37.826218
2,17721851,144.86632,-37.831226
3,17725855,144.86768,-37.827423
4,17745057,144.86790,-37.826270


In [43]:
# Load in tram stop data
tram = pd.read_csv("data\metro_tram_stops.txt")
tram_coords = tram[["stop_id", "stop_lon","stop_lat"]]
tram_coords.head()

,stop_id,stop_lon,stop_lat
0,10311,145.028508,-37.862455
1,10371,145.025382,-37.862069
2,1083,144.898841,-37.769699
3,11285,145.022754,-37.861710
4,1185,145.043375,-37.864226


In [44]:
# Define Haversine distance function
def haversine_vec(lon1, lat1, lon2, lat2):
    """
    Compute distance (meters) between two points using vectorised Haversine formula
    """
    lon1_rad = np.radians(lon1)[:, np.newaxis]
    lat1_rad = np.radians(lat1)[:, np.newaxis]

    lon2_rad = np.radians(lon2)[np.newaxis, :]
    lat2_rad = np.radians(lat2)[np.newaxis, :]

    earth_mean_radius = 6371000
    phi1, phi2 = lat1_rad, lat2_rad
    dphi1 = lat2_rad - lat1_rad
    dlambda = lon2_rad - lon1_rad
    
    a = np.sin(dphi1/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

    return earth_mean_radius * c

In [45]:
# Find top 5 closest tram stops for each rental
dist_matrix = haversine_vec(rental_coords["lon"].values, rental_coords["lat"].values, tram_coords["stop_lon"].values, tram_coords["stop_lat"].values)

top5_indices = np.argsort(dist_matrix, axis=1)[:,:5]

rental_coords["top5_idx"] = list(top5_indices)
rental_coords["top5_tuple"] = [tuple(int(i) for i in sorted(x)) for x in top5_indices]

C:\Users\chinj\AppData\Local\Temp\ipykernel_19492\300191937.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rental_coords["top5_idx"] = list(top5_indices)
C:\Users\chinj\AppData\Local\Temp\ipykernel_19492\300191937.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rental_coords["top5_tuple"] = [tuple(int(i) for i in sorted(x)) for x in top5_indices]


In [46]:
# Group results by their top 5 closest tram stops
groups = rental_coords.groupby("top5_tuple")["listing_id"].apply(list).reset_index()
groups["num_rentals"] = groups["listing_id"].apply(len)

In [ ]:
# API key for OpenRouteService (ORS); hidden here for privacy
API_KEY = NA # Replace NA with your actual key when running

# Initialize ORS client with the API key
client = openrouteservice.Client(key = API_KEY)

In [48]:
# Batch rentals to stay under API route limit
top_n_trams = 5
max_route_per_batch = 3500

batches = []
current_batch = []
current_batch_rentals = 0

# Loop over each group of rentals that share the same top 5 tram stops
for _, row in groups.iterrows():
    group_name = row["top5_tuple"]
    rentals = row["listing_id"]
    group_size = len(rentals)

    start = 0
    # Split group into chunks so that adding them won't exceed ORS route limit
    while start < group_size:
        # Estimate number of rentals allowed to add to current batch without exceeding ORS route limit
        num_groups_if_added = len(current_batch) + 1
        max_rentals_for_this_chunk = max(
            (max_route_per_batch // (num_groups_if_added * top_n_trams)) - current_batch_rentals,
            1
        )

        # Determine slice of rentals for this chunk
        end = min(start + max_rentals_for_this_chunk, group_size)
        chunk = rentals[start:end]

        # Compute estimated number of routes if this chunk is added
        num_origins = current_batch_rentals + len(chunk)
        num_destinations = num_groups_if_added * top_n_trams
        estimated_routes = num_origins * num_destinations

        # Start new batch if adding this chunk exceeds ORS route limit
        if estimated_routes >= max_route_per_batch: 
            if current_batch:
                batches.append(current_batch)
            current_batch = [(group_name, chunk)]     
            current_batch_rentals = len(chunk)
        else:
            # Add chunk to batch otherwise
            current_batch.append((group_name, chunk))
            current_batch_rentals += len(chunk)
        start = end

# Append last batch if there are any remaining rentals
if current_batch:
        batches.append(current_batch)

print(f"Created {len(batches)} batches with full ORS route limit logic.")

Created 151 batches with full ORS route limit logic.


In [49]:
# Print and check summary of batches
for i, batch in enumerate(batches):
    num_groups = len(batch)
    num_rentals = sum(len(rentals) for _, rentals in batch)
    print(f"Batch {i+1}: {num_groups} groups, {num_rentals} rentals")

Batch 1: 16 groups, 43 rentals
Batch 2: 11 groups, 58 rentals
Batch 3: 19 groups, 36 rentals
Batch 4: 20 groups, 33 rentals
Batch 5: 17 groups, 41 rentals
Batch 6: 24 groups, 28 rentals
Batch 7: 21 groups, 33 rentals
Batch 8: 18 groups, 38 rentals
Batch 9: 18 groups, 38 rentals
Batch 10: 14 groups, 47 rentals
Batch 11: 13 groups, 53 rentals
Batch 12: 12 groups, 54 rentals
Batch 13: 16 groups, 43 rentals
Batch 14: 12 groups, 55 rentals
Batch 15: 19 groups, 30 rentals
Batch 16: 6 groups, 69 rentals
Batch 17: 4 groups, 120 rentals
Batch 18: 3 groups, 233 rentals
Batch 19: 4 groups, 160 rentals
Batch 20: 9 groups, 75 rentals
Batch 21: 4 groups, 123 rentals
Batch 22: 3 groups, 204 rentals
Batch 23: 19 groups, 34 rentals
Batch 24: 1 groups, 1 rentals
Batch 25: 1 groups, 349 rentals
Batch 26: 7 groups, 88 rentals
Batch 27: 12 groups, 58 rentals
Batch 28: 4 groups, 62 rentals
Batch 29: 3 groups, 210 rentals
Batch 30: 9 groups, 77 rentals
Batch 31: 8 groups, 87 rentals
Batch 32: 4 groups, 150 r

In [50]:
# Call ORS to get distance to closest tram stops
all_rental_ids = []
all_distances = []
all_tram_ids = []

for batch in tqdm(batches, desc="Processing batches"):
    batch_rentals = [r for _, rentals in batch for r in rentals]
    batch_coords = rental_coords.set_index("listing_id").loc[batch_rentals][["lon", "lat"]].values.tolist()

    batch_tram_indices = [
        rental_coords.loc[rental_coords["listing_id"]==r, "top5_idx"].values[0] for r in batch_rentals
    ]

    unique_tram_indices = list(set(idx for indices in batch_tram_indices for idx in indices))

    origins = batch_coords
    destinations = tram_coords.loc[tram_coords.index.isin(unique_tram_indices)][["stop_lon", "stop_lat"]].values.tolist()
    dest_ids_map = {idx: tram_coords["stop_id"].iloc[idx] for idx in unique_tram_indices}
   
    tram_pos_map = {idx: i for i, idx in enumerate(unique_tram_indices)}

    num_origins = len(origins)
    num_destinations = len(destinations)
    num_routes = num_origins * num_destinations
    print(f"Checking batch → {num_origins} origins × {num_destinations} destinations = {num_routes} routes")

    # Call ORS distance matrix
    matrix = client.distance_matrix(
        locations = origins + destinations,
        profile = "driving-car",
        metrics = ["distance"],
        sources = list(range(len(origins))),
        destinations = list(range(len(origins), len(origins)+len(destinations)))
    )

    # Extract distances and tram stop IDs for top 5 tram stops
    for i, rental in enumerate(batch_rentals):
        top_indices = batch_tram_indices[i]
        dest_positions = [tram_pos_map[idx] for idx in top_indices]
        rental_distances = [matrix["distances"][i][pos] for pos in dest_positions]
        rental_tram_ids = [dest_ids_map[idx] for idx in top_indices]

        sorted_idx = np.argsort(rental_distances)
        top_n_distances = [rental_distances[k] for k in sorted_idx]
        top_n_ids = [rental_tram_ids[k] for k in sorted_idx]

        all_rental_ids.append(batch_rentals[i])
        all_distances.append(top_n_distances)
        all_tram_ids.append(top_n_ids)

Processing batches:   0%|          | 0/151 [00:00<?, ?it/s]

Checking batch → 43 origins × 40 destinations = 1720 routes


Processing batches:   1%|          | 1/151 [00:01<04:51,  1.94s/it]

Checking batch → 58 origins × 24 destinations = 1392 routes


Processing batches:   1%|▏         | 2/151 [00:08<11:37,  4.68s/it]

...

Processing batches:  99%|█████████▊| 149/151 [06:20<00:00,  2.36it/s]

Checking batch → 36 origins × 21 destinations = 756 routes


c:\Users\chinj\anaconda3\Lib\site-packages\openrouteservice\client.py:211: UserWarning: Rate limit exceeded. Retrying for the 1st time.
  warnings.warn('Rate limit exceeded. Retrying for the {0}{1} time.'.format(retry_counter + 1,
Processing batches:  99%|█████████▉| 150/151 [06:22<00:00,  1.44it/s]

Checking batch → 26 origins × 28 destinations = 728 routes


c:\Users\chinj\anaconda3\Lib\site-packages\openrouteservice\client.py:211: UserWarning: Rate limit exceeded. Retrying for the 1st time.
  warnings.warn('Rate limit exceeded. Retrying for the {0}{1} time.'.format(retry_counter + 1,
Processing batches: 100%|██████████| 151/151 [06:24<00:00,  2.54s/it]


In [51]:
# Save ORS distances for tram stops into dataframe
df = pd.DataFrame({
    "rental_id": all_rental_ids,
    "top_distances": all_distances,
    "top_tram_ids": all_tram_ids
})

df.head()

,rental_id,top_distances,top_tram_ids
0,17745632,"[85.48, 125.58, 130.44, 1160.58, 1385.31]","[950, 9156, 10311, 20528, 20529]"
1,17682583,"[85.48, 125.58, 130.44, 1160.58, 1385.31]","[950, 9156, 10311, 20528, 20529]"
2,17640056,"[115.94, 378.13, 428.06, 20694.89, 23743.12]","[18487, 992, 18486, 1083, 18485]"
3,17727285,"[468.24, 730.43, 780.36, 21047.19, 24095.41]","[18487, 992, 18486, 1083, 18485]"
4,17674738,"[435.43, 627.47, 939.6, 11287.8, 21206.42]","[18488, 18487, 18486, 4945, 1083]"


In [52]:
# Save final CSV
df.to_csv("rentals_distances_to_tram_stops.csv", index=False)